<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Quantize_Weights_and_Activations_for_Memory_Efficient_LLMs_with_llm_compressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Make LLMs Faster and Lighter with W8A8 Quantization](https://kaitchup.substack.com/p/make-llms-faster-and-lighter-with)*

This notebooks shows how to quantize weights and activations of LLMs with llm-compressor. Qwen2.5 7B is used for example.
Most of the code is [modified code examples from the llm-compressor project](https://github.com/vllm-project/llm-compressor/tree/main/examples/quantization_w8a8_fp8).

# Installation

*Note: lm-eval, vllm, langdetect, and immutabledict, are only used for evaluation in this notebook*

In [ ]:
!pip install --upgrade transformers llmcompressor datasets lm-eval vllm langdetect immutabledict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://

# Load the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# W8A8-INT8

## Calibration Dataset

In [ ]:
from datasets import load_dataset

NUM_CALIBRATION_SAMPLES=512
MAX_SEQUENCE_LENGTH=2048

# Load dataset.
ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

# Preprocess the data into the format the model is trained with.
def preprocess(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False,)}
ds = ds.map(preprocess)

# Tokenize the data (be careful with bos tokens - we need add_special_tokens=False since the chat_template already added it).
def tokenize(sample):
    return tokenizer(sample["text"], padding=False, max_length=MAX_SEQUENCE_LENGTH, truncation=True, add_special_tokens=False)
ds = ds.map(tokenize, remove_columns=ds.column_names)

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


(…)-00000-of-00003-a3ecf92756993583.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


(…)-00001-of-00003-0a1804bcb6ae68c6.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

...

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

## Quantization

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

# Configure the quantization algorithms to run.
recipe = [
    SmoothQuantModifier(smoothing_strength=0.8),
    GPTQModifier(targets="Linear", scheme="W8A8", ignore=["lm_head"]),
]

# Apply quantization.
oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# Save to disk compressed.
SAVE_DIR = MODEL_ID.split("/")[1] + "-W8A8-Dynamic-Per-Token"
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)

2025-04-17T13:53:07.553689+0000 | reset | INFO - Compression lifecycle reset
2025-04-17T13:53:07.555212+0000 | from_modifiers | INFO - Creating recipe from modifiers
2025-04-17T13:53:07.583011+0000 | _infer_mappings_from_model | INFO - No SmoothQuantModifier.mappings provided, inferring from model...
2025-04-17T13:53:08.444938+0000 | _calibrate | INFO - Running SmoothQuantModifier calibration with 512 samples...


100%|██████████| 512/512 [03:08<00:00,  2.72it/s]

2025-04-17T13:56:16.714015+0000 | _apply_smoothing | INFO - Smoothing activation scales...


2025-04-17T13:56:17.236330+0000 | _check_build_quant_modifier | WARNING - GPTQ quantization is set to True without an active quantization modifier.
2025-04-17T13:56:17.237025+0000 | _build_quant_modifier | INFO - Building quantization modifier with args: {'targets': 'Linear', 'scheme': 'W8A8', 'ignore': ['lm_head']}
2025-04-17T13:56:17.273229+0000 | _check_calibration_data | INFO - Skipping QuantizationModifier calibration, it is not required for the provided quantization config.


(1/29): Calibrating: 100%|██████████| 512/512 [01:07<00:00,  7.64it/s]

...

2025-04-17T14:43:17.136469+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2025-04-17T14:43:17.136919+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide


Checking whether model follows 2:4 sparsity structure: 100%|██████████| 197/197 [00:18<00:00, 10.61it/s]


2025-04-17T14:45:18.759567+0000 | get_model_compressor | INFO - Inferring a sparsity configuration requires a global sparsity calculation. This can be costly for large models. To skip the calculation of compression statistics set skip_compression_stats=True


Calculating model sparsity: 100%|██████████| 731/731 [00:16<00:00, 45.59it/s]
Calculating quantization compression ratio: 284it [00:00, 431.65it/s]
Quantized Compression: 100%|██████████| 731/731 [00:13<00:00, 55.19it/s]


('Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/tokenizer_config.json',
 'Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/special_tokens_map.json',
 'Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/vocab.json',
 'Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/merges.txt',
 'Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/added_tokens.json',
 'Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token/tokenizer.json')

## Evaluation
When I ran this with vLLM, it didn't seem to work with the most recent version. I had to downgrade to 0.7.0.

In [ ]:
!pip install vllm==0.7.0

  Using cached vllm-0.7.0-cp38-abi3-manylinux1_x86_64.whl.metadata (12 kB)
  Using cached blake3-1.0.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached lm_format_enforcer-0.10.11-py3-none-any.whl.metadata (17 kB)
  Using cached outlines-0.1.11-py3-none-any.whl.metadata (17 kB)
  Using cached xgrammar-0.1.18-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.1/264.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [ ]:
!lm_eval --model vllm \
  --model_args pretrained="./Qwen2.5-7B-Instruct-W8A8-Dynamic-Per-Token" \
  --tasks leaderboard_ifeval \
  --apply_chat_template \
  --batch_size 'auto'

2025-04-17 15:05:34.431822: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-17 15:05:34.450962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744902334.474290   21970 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744902334.483036   21970 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-17 15:05:34.508758: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

# W8A8-FP8

## Quantization

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

# Configure the simple PTQ quantization
recipe = QuantizationModifier(
  targets="Linear", scheme="FP8_DYNAMIC", ignore=["lm_head"])

# Apply the quantization algorithm.
oneshot(model=model, recipe=recipe)

# Save the model.
SAVE_DIR = MODEL_ID.split("/")[1] + "-FP8-Dynamic"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

2025-04-17T15:24:15.127846+0000 | reset | INFO - Compression lifecycle reset
2025-04-17T15:24:15.129334+0000 | from_modifiers | INFO - Creating recipe from modifiers


manager stage: Modifiers initialized


2025-04-17T15:24:15.461765+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers


manager stage: Modifiers finalized


2025-04-17T15:24:15.463512+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2025-04-17T15:24:15.464096+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide


Checking whether model follows 2:4 sparsity structure: 100%|██████████| 197/197 [00:17<00:00, 11.45it/s]


2025-04-17T15:26:17.075848+0000 | get_model_compressor | INFO - Inferring a sparsity configuration requires a global sparsity calculation. This can be costly for large models. To skip the calculation of compression statistics set skip_compression_stats=True


Calculating model sparsity: 100%|██████████| 731/731 [00:05<00:00, 128.41it/s]
Calculating quantization compression ratio: 284it [00:00, 459.34it/s]
Quantized Compression: 100%|██████████| 731/731 [00:20<00:00, 36.33it/s]


('Qwen2.5-7B-Instruct-FP8-Dynamic/tokenizer_config.json',
 'Qwen2.5-7B-Instruct-FP8-Dynamic/special_tokens_map.json',
 'Qwen2.5-7B-Instruct-FP8-Dynamic/vocab.json',
 'Qwen2.5-7B-Instruct-FP8-Dynamic/merges.txt',
 'Qwen2.5-7B-Instruct-FP8-Dynamic/added_tokens.json',
 'Qwen2.5-7B-Instruct-FP8-Dynamic/tokenizer.json')

## Evaluation

In [ ]:
!pip install --upgrade vllm

  Using cached vllm-0.8.4-cp38-abi3-manylinux1_x86_64.whl.metadata (27 kB)
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached llguidance-0.7.16-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.6 kB)
  Using cached compressed_tensors-0.9.3-py3-none-any.whl.metadata (7.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.1/294.1 MB 3.7 MB/s eta 0:00:00
Using cached compressed_tensors-0.9.3-py3-none-any.whl (98 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 92

In [ ]:
!VLLM_USE_V1=0 lm_eval --model vllm \
  --model_args pretrained="./Qwen2.5-7B-Instruct-FP8-Dynamic" \
  --tasks leaderboard_ifeval \
  --apply_chat_template \
  --batch_size 'auto'

2025-04-17 15:38:19.967640: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-17 15:38:19.986755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744904300.009845   30950 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744904300.016883   30950 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-17 15:38:20.039875: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

# Qwen2.5-7B Instruct Evaluation (baseline)

To get a point of comparison, this evaluates Qwen2.5 without quantization.

In [ ]:
!VLLM_USE_V1=0 lm_eval --model vllm \
  --model_args pretrained="Qwen/Qwen2.5-7B-Instruct",max_model_len=20000 \
  --tasks leaderboard_ifeval \
  --apply_chat_template \
  --batch_size 'auto'

2025-04-17 15:27:44.241752: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-17 15:27:44.260359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744903664.282378   27864 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744903664.289253   27864 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-17 15:27:44.311883: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

# requirements.txt (if you have issues with recent versions)

In [ ]:
!pip freeze

absl-py==1.4.0
accelerate==1.5.2
aiohappyeyeballs==2.6.1
aiohttp==3.11.15
aiosignal==1.3.2
alabaster==1.0.0
albucore==0.0.23
albumentations==2.0.5
ale-py==0.10.2
altair==5.5.0
annotated-types==0.7.0
anyio==4.9.0
argon2-cffi==23.1.0
argon2-cffi-bindings==21.2.0
array_record==0.7.1
arviz==0.21.0
astropy==7.0.1
astropy-iers-data==0.2025.4.7.0.35.30
astunparse==1.6.3
atpublic==5.1
attrs==25.3.0
audioread==3.0.1
autograd==1.7.0
babel==2.17.0
backcall==0.2.0
backports.tarfile==1.2.0
beautifulsoup4==4.13.3
betterproto==2.0.0b6
bigframes==1.42.0
bigquery-magics==0.9.0
bleach==6.2.0
blinker==1.9.0
blis==1.3.0
blosc2==3.3.0
bokeh==3.6.3
Bottleneck==1.4.2
bqplot==0.12.44
branca==0.8.1
CacheControl==0.14.2
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.1.31
cffi==1.17.1
chardet==5.2.0
charset-normalizer==3.4.1
chex==0.1.89
clarabel==0.10.0
click==8.1.8
cloudpathlib==0.21.0
cloudpickle==3.1.1
cmake==3.31.6
cmdstanpy==1.2.5
colorama==0.4.6
colorcet==3.1.0
colorlover==0.3.0
colour==0.1.5
community